# Step 1 Statistical test

# Phase 3 - Statistical testing and logistic regression
# Step 1 - Statistical Testing

In [20]:
import pandas as pd 
import numpy as np
from scipy.stats import chi2_contingency
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

In [16]:
df = pd.read_csv('cleaned_df.csv')

In [17]:
cat_cols = ['Complain','PreferredLoginDevice','PreferredPaymentMode',
            'Gender','PreferedOrderCat','MaritalStatus','CityTier']

for col in cat_cols:
    ct = pd.crosstab(df[col], df['Churn'])
    chi2, p, dof, _ = chi2_contingency(ct)
    print(f"{col}: chi2={chi2:.2f}, p={p:.4f}")

Complain: chi2=350.93, p=0.0000
PreferredLoginDevice: chi2=73.54, p=0.0000
PreferredPaymentMode: chi2=77.90, p=0.0000
Gender: chi2=4.66, p=0.0308
PreferedOrderCat: chi2=288.64, p=0.0000
MaritalStatus: chi2=188.67, p=0.0000
CityTier: chi2=40.98, p=0.0000


All 7 categorical variables passed the chi-square test (p < 0.05), meaning all of these variables have a statistically significant association with churn

    Very strong singla (chi2 > 200)
    -- Complain (350.93): The single strongest driver. Customers who complain churn at nearly 3x the rate of those who do not.
    -- PreferedOrderCat (288.64): The category a customer orders from is strongly tied to churn. Some categories are much higher risk.
    -- MaritalStatus (188.67): Marital status segments customers into meaningfully different churn risk groups.

    Moderate signal (chi2 = 40 to 80)
    -- PreferredLoginDevice (73.54): How customers log in separates churn behavior. Mobile-only users often show higher churn in this dataset.
    -- PreferredPaymentMode (77.90): Payment method choice correlates with churn, possibly reflecting engagement depth or trust.
    -- CityTier (40.98): Tier 2 and Tier 3 city customers churn more, which aligns with your EDA findings.

    Weak but significant signal
    -- Gender (4.66, p=0.03): Statistically significant but practically weak. The association exists but chi2 is very low compared to others.Dropping it from the model would be good to keep the model clean.

In [18]:
cat_cols.remove('Gender')

In [19]:
num_cols = ['Tenure', 'WarehouseToHome', 'HourSpendOnApp', 'NumberOfDeviceRegistered',
            'SatisfactionScore', 'NumberOfAddress', 'OrderAmountHikeFromlastYear',
            'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']

churned = df[df['Churn'] == 1]
not_churned = df[df['Churn'] == 0]

for col in num_cols:
    stat, p = mannwhitneyu(churned[col], not_churned[col], alternative='two-sided')
    print(f"{col}: U={stat:.2f}, p={p:.4f}")

Tenure: U=888865.00, p=0.0000
WarehouseToHome: U=2494375.50, p=0.0000
HourSpendOnApp: U=2259450.00, p=0.3436
NumberOfDeviceRegistered: U=2546899.00, p=0.0000
SatisfactionScore: U=2568666.00, p=0.0000
NumberOfAddress: U=2316556.50, p=0.0305
OrderAmountHikeFromlastYear: U=2150234.00, p=0.1287
CouponUsed: U=2160196.50, p=0.1794
OrderCount: U=2104315.00, p=0.0087
DaySinceLastOrder: U=1632258.00, p=0.0000
CashbackAmount: U=1628505.00, p=0.0000


Churn is linked more to customer tenure, app usage pattern, satisfaction, address complexity, recency of last order, device registration, and cashback behavior than to spend increase or coupon use.

Keeping these variables for logistic regression:

    Tenure.
    WarehouseToHome.
    NumberOfDeviceRegistered.
    SatisfactionScore.
    NumberOfAddress.
    DaySinceLastOrder.
    CashbackAmount.
    OrderCount.



# Step 2 Logistic regresssion

In [22]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

impute_cols = [
    'Tenure', 'WarehouseToHome', 'HourSpendOnApp',
    'NumberOfDeviceRegistered', 'SatisfactionScore',
    'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear',
    'CouponUsed', 'OrderCount', 'DaySinceLastOrder',
    'CashbackAmount', 'CityTier'
]

for col in impute_cols:
    df[col + '_missing'] = df[col].isna().astype(int)

imp = IterativeImputer(random_state=42, max_iter=10, sample_posterior=True)
df[impute_cols] = imp.fit_transform(df[impute_cols])

In [24]:
# Use one imputed version (seed=42 is standard)
df_model = df.copy()

# Select your confirmed significant features + missingness flags
features = ['Complain', 'Tenure', 'SatisfactionScore', 'CityTier',
            'DaySinceLastOrder', 'OrderCount', 'HourSpendOnApp',
            'WarehouseToHome_missing', 'Tenure_missing']

X = df_model[features]
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print(classification_report(y_test, model.predict(X_test_scaled)))
print("ROC-AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))

              precision    recall  f1-score   support

           0       0.88      0.95      0.92       936
           1       0.63      0.38      0.48       190

    accuracy                           0.86      1126
   macro avg       0.76      0.67      0.70      1126
weighted avg       0.84      0.86      0.84      1126

ROC-AUC: 0.8309632253711201




The logistic regression model performed well on class 0, with precision of 0.88, recall of 0.95, and F1 score of 0.92. This means the model correctly identified most non churned customers. For class 1, precision was 0.63, recall was 0.38, and F1 score was 0.48. This means the model missed a large share of churned customers, so it is weaker at detecting at risk customers. The support shows that 936 customers belonged to class 0 and 190 belonged to class 1, so the dataset is imbalanced toward non churned customers. Overall accuracy was 0.86, but the ROC AUC of 0.83 shows the model has good separation ability. For a churn project, recall for class 1 matters more, so the model needs improvement if the goal is to catch more churners.

    Moving to model improvement next with class balancing

In [25]:
model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.94      0.72      0.82       936
           1       0.37      0.78      0.50       190

    accuracy                           0.73      1126
   macro avg       0.65      0.75      0.66      1126
weighted avg       0.85      0.73      0.77      1126

ROC-AUC: 0.8310138326585694


The balanced logistic regression model is better suited for churn prediction because it identifies a much larger share of churned customers. Class 1 recall improved to 0.78, which means the model catches most at risk customers. However, class 1 precision is only 0.37, so many flagged customers will not actually churn. Overall accuracy is 0.73 and ROC AUC is 0.83, which shows the model separates churned and non churned customers well. For a retention use case, this is a useful tradeoff because missing a churner is more costly than contacting a few extra customers.

     Next step is to extract coeffecient and odd ratios and rank the strongest churn drivers; to write a short business interpretation for each top feature

In [26]:
coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_[0],
    'Odds_Ratio': np.exp(model.coef_[0])
}).sort_values('Odds_Ratio', ascending=False)

print(coef_df)

                   Feature  Coefficient  Odds_Ratio
0                 Complain     0.677158    1.968277
5               OrderCount     0.466233    1.593979
2        SatisfactionScore     0.369228    1.446618
3                 CityTier     0.247075    1.280275
6           HourSpendOnApp     0.036194    1.036857
7  WarehouseToHome_missing     0.000000    1.000000
8           Tenure_missing     0.000000    1.000000
4        DaySinceLastOrder    -0.503035    0.604692
1                   Tenure    -1.373189    0.253298


The logistic regression results show that complaint behavior is the strongest positive predictor of churn, followed by order count, satisfaction score, and city tier. Tenure has the strongest negative effect, which means long term customers are much less likely to churn. Hour spend on app and missingness flags contribute little to the model. Overall, the model suggests that churn is driven more by experience and relationship depth than by app usage alone.

    What the model says

    -- Complain is the strongest positive driver of churn.
    -- OrderCount and SatisfactionScore also push churn upward in the model.
    -- CityTier increases churn risk.
    -- Tenure strongly reduces churn risk.

# Report line

    The logistic regression model shows that complaints are the strongest churn driver, while longer tenure protects against churn. Order count, satisfaction score, and city tier also matter, which suggests churn is tied more to customer experience and relationship depth than to app usage alone.

# Phase 3 ends here

In [27]:
df_model.to_csv("phase3_model_data.csv", index=False)
print("Model dataset saved as phase3_model_data.csv")

Model dataset saved as phase3_model_data.csv
